# Module 3 : Hospital KPI Engineering

### Calculate:
- Total Admissions
- Occupancy Rate
- Average Length of Stay
- Readmission Rate
- Bed Utilization Rate
- Department Efficiency Score

### Deliverables
- hospital_final_dataset.xlsx
- generate_hospital_kpis.ipynb

In [1]:
# Import required libraries
import pandas as pd

# Load the clean dataset
df = pd.read_csv('hospital_cleaned.csv')
df.head(5)

,Hospital ID,Hospital Name,Hospital Type,City,State,Department,Department ID,Doctor,Nurses,Staff Count,...,Transferred,Transfer_From_Department,Transfer_To_Department,Transfer_Date,Number_of_Transfers,Dept_ICU_Bed_Capacity_Derived,Dept_Staff_Capacity_Derived,Admissions_Rate_%_Derived,Staff_Utilization_%_Derived,Bed_Occupancy_Rate_%
0,HOSP00016,C K Hospital,Private,Mumbai,Maharashtra,General Medicine,DEPT004,Dr. Arjun Sharma,12,257,...,No,General Medicine,Not Transferred,Not Applicable,0,98,598,0.0,43.0,0.2
1,HOSP00019,Cure Well Hospital,Private,Bengaluru,Karnataka,General Surgery,DEPT010,Dr. Sneha Rao,15,288,...,No,General Surgery,Not Transferred,Not Applicable,0,100,583,0.2,49.4,0.2
2,HOSP00004,Yashoda Hospital,Private,Hyderabad,Telangana,Pulmonology,DEPT005,Dr. Priya Reddy,20,181,...,No,Pulmonology,Not Transferred,Not Applicable,0,97,590,0.0,30.7,0.2
3,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,Pulmonology,DEPT005,Dr. Rahul Verma,23,170,...,Yes,Pulmonology,Cardiology,9/17/2025,1,99,597,0.2,28.5,0.4
4,HOSP00017,Dharma Hospital,Private,Delhi,Delhi,ICU,DEPT009,Dr. Priya Reddy,34,116,...,No,ICU,Not Transferred,Not Applicable,0,100,576,0.0,20.1,0.6


#### Clean the Date Formate

In [2]:
# Date formate clining
df['Admission Date'] = pd.to_datetime(df['Admission Date'].str.replace('-', '/'), format='%m/%d/%Y')

df['Discharge Date'] = pd.to_datetime(df['Discharge Date'].str.replace('-', '/'), format='%m/%d/%Y')

### KPI Calculation

### 1. Total Admission

In [3]:
# Calculate total admission

total_admissions = len(df)
total_admissions

10000

### 2. Bed occupancy Rate

In [5]:
# addmission where the assigned bed is marked occupied
occupancy_rate = (df['Bed Occupied'] == 'Yes').mean() * 100
occupancy_rate

np.float64(50.36000000000001)

### 3. Average length of Stay

In [6]:
# Average of length of stay

avg_length_of_stay = df['Length of Stay'].mean()
avg_length_of_stay

np.float64(8.0684)

### 4. Readmission Rate

In [7]:
# patient that admit repeat time

readmission_rate = (df['Readmission'] == 'Yes').mean() * 100
readmission_rate

np.float64(49.87)

### 5. Bed Utilization Rate

In [8]:
df['Bed_Utilization_Ratio'] = df['Beds_Occupied_Count'] / df['Beds Available'] * 100
bed_utilization_rate = df['Bed_Utilization_Ratio'].mean()
bed_utilization_rate

np.float64(1.25701089849642)

### 6. Department Effiency Score

In [10]:
'''Composite score (0-100) per department:
- 30% Occupancy Rate
- 30% (100 - Readmission Rate)
- 20% Staff Utilization
- 20% shorter Average Length of Stay (vs 15-day max in this dataset)'''

dept_kpis = df.groupby('Department').agg(
    Total_Admissions=('Patient ID', 'count'),
    Occupancy_Rate=('Bed Occupied', lambda x: (x == 'Yes').mean() * 100),
    Avg_Length_of_Stay=('Length of Stay', 'mean'),
    Readmission_Rate=('Readmission', lambda x: (x == 'Yes').mean() * 100),
    Bed_Utilization_Rate=('Bed_Utilization_Ratio', 'mean'),
    Staff_Utilization=('Staff_Utilization_%_Derived', 'mean'),
).reset_index()

dept_kpis['Department_Efficiency_Score'] = (
    dept_kpis['Occupancy_Rate'] * 0.30
    + (100 - dept_kpis['Readmission_Rate']) * 0.30
    + dept_kpis['Staff_Utilization'] * 0.20
    + ((15 - dept_kpis['Avg_Length_of_Stay']) / 15 * 100) * 0.20
)

dept_kpis

,Department,Total_Admissions,Occupancy_Rate,Avg_Length_of_Stay,Readmission_Rate,Bed_Utilization_Rate,Staff_Utilization,Department_Efficiency_Score
0,Cardiology,991,51.059536,7.874874,51.160444,1.217002,58.548133,51.179522
1,General Medicine,964,53.941909,7.946058,50.000000,1.192844,57.357573,52.059343
2,General Surgery,1020,49.901961,8.069608,51.176471,1.274107,57.331078,50.324386
3,ICU,1010,51.683168,8.026733,47.029703,1.186259,59.173168,52.528363
4,Nephrology,999,49.549550,8.084084,49.749750,1.192524,56.628428,50.486847
5,Neurology,1033,50.048403,8.225557,47.241045,1.264535,57.985866,51.471972
6,Oncology,1031,52.279340,8.272551,51.988361,1.323986,58.909796,50.839185
7,Orthopedics,958,48.329854,8.007307,49.686848,1.301729,56.950522,50.306597
8,Psychiatry,1016,47.244094,8.021654,50.196850,1.295807,58.146260,50.047887
9,Pulmonology,978,49.591002,8.138037,50.511247,1.319249,58.128528,50.498916


### Overall KPIs Summary

In [11]:
summary = pd.DataFrame({
    'KPI': [
        'Total Admissions',
        'Occupancy Rate (%)',
        'Average Length of Stay (days)',
        'Readmission Rate (%)',
        'Bed Utilization Rate (%)',
        'Department Efficiency Score (avg)',
    ],
    'Value': [
        total_admissions,
        occupancy_rate,
        avg_length_of_stay,
        readmission_rate,
        bed_utilization_rate,
        dept_kpis['Department_Efficiency_Score'].mean(),
    ],
})
summary

,KPI,Value
0,Total Admissions,10000.000000
1,Occupancy Rate (%),50.360000
2,Average Length of Stay (days),8.068400
3,Readmission Rate (%),49.870000
4,Bed Utilization Rate (%),1.257011
5,Department Efficiency Score (avg),50.974302


#### Export Final Dataset

In [12]:
with pd.ExcelWriter('hospital_final_dataset.xlsx', engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='KPI_Summary', index=False)
    dept_kpis.to_excel(writer, sheet_name='Department_KPIs', index=False)
    df.drop(columns=['Bed_Utilization_Ratio']).to_excel(writer, sheet_name='Raw_Data', index=False)

print('saved hospital_final_dataset.xlsx')

saved hospital_final_dataset.xlsx
